# Automated Machine Learning for Regression Tasks

This notebook demonstrates automated machine learning workflows for regression problems using various algorithms and techniques. It's designed to work seamlessly in Google Colab.

## Features:
- Automated data preprocessing
- Multiple regression algorithms comparison
- Hyperparameter tuning
- Model evaluation and visualization
- Easy-to-use interface for different datasets

## 1. Setup and Installation
First, let's install required packages and import necessary libraries.

In [ ]:
# Install required packages for Google Colab
!pip install scikit-learn pandas numpy matplotlib seaborn plotly
!pip install xgboost lightgbm
!pip install scipy

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regression algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
import lightgbm as lgb

# Evaluation metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error
)

# Additional utilities
from scipy import stats
import joblib
import pickle

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported successfully!")

## 2. Data Loading and Exploration
Load your dataset and perform initial exploration.

In [ ]:
# Option 1: Load demo dataset
from sklearn.datasets import load_boston, load_diabetes, fetch_california_housing

# Choose one of the demo datasets
dataset_choice = 'california_housing'  # Options: 'diabetes', 'california_housing'

if dataset_choice == 'diabetes':
    data = load_diabetes()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    target_name = 'Disease Progression'
elif dataset_choice == 'california_housing':
    data = fetch_california_housing()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    target_name = 'House Value (hundreds of thousands of dollars)'

print(f"Dataset: {dataset_choice}")
print(f"Shape: {df.shape}")
print(f"Target: {target_name}")
df.head()

In [ ]:
# Option 2: Upload your own CSV file
# Uncomment the following lines to upload your own dataset

# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(filename)
# 
# # Specify your target column name
# target_column = 'target'  # Replace with your target column name
# target_name = 'Your Target Variable'
# 
# print(f"Dataset shape: {df.shape}")
# print("Columns:", df.columns.tolist())
# df.head()

In [ ]:
# Data exploration
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum().sum())
print(f"\nTarget statistics:")
print(df['target'].describe())

# Basic statistics
df.describe()

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Target distribution histogram
axes[0,0].hist(df['target'], bins=50, alpha=0.7, color='skyblue')
axes[0,0].set_title('Target Distribution')
axes[0,0].set_xlabel(target_name)
axes[0,0].set_ylabel('Frequency')

# Target distribution box plot
axes[0,1].boxplot(df['target'])
axes[0,1].set_title('Target Box Plot')
axes[0,1].set_ylabel(target_name)

# Q-Q plot for normality check
stats.probplot(df['target'], dist="norm", plot=axes[1,0])
axes[1,0].set_title('Q-Q Plot (Normality Check)')

# Target vs first feature scatter plot
first_feature = df.columns[0]
axes[1,1].scatter(df[first_feature], df['target'], alpha=0.5)
axes[1,1].set_xlabel(first_feature)
axes[1,1].set_ylabel(target_name)
axes[1,1].set_title(f'{target_name} vs {first_feature}')

plt.tight_layout()
plt.show()

# Correlation heatmap for numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 1:
    plt.figure(figsize=(12, 8))
    correlation_matrix = df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
    plt.title('Feature Correlation Heatmap')
    plt.show()
    
    # Show top correlations with target
    target_corr = correlation_matrix['target'].abs().sort_values(ascending=False)
    print("\nTop correlations with target:")
    print(target_corr.head(10))

## 3. Automated Data Preprocessing
Prepare the data for machine learning with automated preprocessing steps.

In [ ]:
class AutomatedRegressionPreprocessor:
    def __init__(self):
        self.preprocessor = None
        self.target_transformer = None
        
    def fit_transform(self, X, y):
        # Separate numerical and categorical features
        numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
        categorical_features = X.select_dtypes(include=['object']).columns.tolist()
        
        print(f"Numerical features: {numerical_features}")
        print(f"Categorical features: {categorical_features}")
        
        # Create preprocessing pipelines
        numerical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])
        
        # Combine preprocessors
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', numerical_transformer, numerical_features),
                ('cat', categorical_transformer, categorical_features)
            ]
        )
        
        # Fit and transform features
        X_processed = self.preprocessor.fit_transform(X)
        
        # Check if target needs transformation (log transform for skewed data)
        skewness = stats.skew(y)
        print(f"\nTarget skewness: {skewness:.3f}")
        
        if abs(skewness) > 1:  # Highly skewed
            print("Applying log transformation to target due to high skewness")
            y_processed = np.log1p(y)  # log1p for non-negative values
            self.target_transformer = 'log'
        else:
            y_processed = y
            self.target_transformer = None
            
        return X_processed, y_processed
    
    def transform(self, X, y=None):
        X_processed = self.preprocessor.transform(X)
        
        if y is not None and self.target_transformer == 'log':
            y_processed = np.log1p(y)
            return X_processed, y_processed
        return X_processed
    
    def inverse_transform_target(self, y_transformed):
        """Transform target back to original scale"""
        if self.target_transformer == 'log':
            return np.expm1(y_transformed)
        return y_transformed

# Prepare features and target
X = df.drop('target', axis=1)
y = df['target']

# Apply preprocessing
preprocessor = AutomatedRegressionPreprocessor()
X_processed, y_processed = preprocessor.fit_transform(X, y)

print(f"\nOriginal shape: {X.shape}")
print(f"Processed shape: {X_processed.shape}")
print(f"Target range: [{y_processed.min():.3f}, {y_processed.max():.3f}]")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training target range: [{y_train.min():.3f}, {y_train.max():.3f}]")
print(f"Test target range: [{y_test.min():.3f}, {y_test.max():.3f}]")

## 4. Automated Model Training and Comparison
Train multiple regression models and compare their performance.

In [ ]:
# Define regression models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(random_state=42),
    'Lasso Regression': Lasso(random_state=42, max_iter=2000),
    'Elastic Net': ElasticNet(random_state=42, max_iter=2000),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100),
    'SVR': SVR(),
    'K-Nearest Neighbors': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42),
    'LightGBM': lgb.LGBMRegressor(random_state=42, verbose=-1)
}

print(f"Total models to train: {len(models)}")

In [ ]:
# Train and evaluate all models
results = {}
cv_scores = {}

# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Training models...\n")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Cross-validation (using negative MSE, converted to positive)
    cv_score = -cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
    cv_scores[name] = cv_score
    
    # Train on full training set
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    
    # Transform predictions back to original scale if needed
    y_test_original = preprocessor.inverse_transform_target(y_test)
    y_pred_original = preprocessor.inverse_transform_target(y_pred)
    
    # Calculate metrics on original scale
    mse = mean_squared_error(y_test_original, y_pred_original)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_original, y_pred_original)
    r2 = r2_score(y_test_original, y_pred_original)
    
    # MAPE (Mean Absolute Percentage Error) - handle division by zero
    try:
        mape = mean_absolute_percentage_error(y_test_original, y_pred_original)
    except:
        mape = np.mean(np.abs((y_test_original - y_pred_original) / np.maximum(np.abs(y_test_original), 1e-8))) * 100
    
    results[name] = {
        'model': model,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2_score': r2,
        'mape': mape,
        'cv_mean': cv_score.mean(),
        'cv_std': cv_score.std(),
        'predictions': y_pred_original,
        'predictions_transformed': y_pred
    }
    
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  CV RMSE: {np.sqrt(cv_score.mean()):.4f} (+/- {np.sqrt(cv_score.std()) * 2:.4f})")
    print()

print("All models trained successfully!")

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'RMSE': [results[model]['rmse'] for model in results.keys()],
    'MAE': [results[model]['mae'] for model in results.keys()],
    'R² Score': [results[model]['r2_score'] for model in results.keys()],
    'MAPE (%)': [results[model]['mape'] for model in results.keys()],
    'CV RMSE': [np.sqrt(results[model]['cv_mean']) for model in results.keys()],
    'CV Std': [np.sqrt(results[model]['cv_std']) for model in results.keys()]
})

# Sort by R² score (higher is better)
results_df = results_df.sort_values('R² Score', ascending=False)
print("Model Performance Comparison:")
print(results_df.round(4))

# Find best model
best_model_name = results_df.iloc[0]['Model']
best_model = results[best_model_name]['model']
print(f"\nBest Model: {best_model_name}")
print(f"Best R² Score: {results_df.iloc[0]['R² Score']:.4f}")
print(f"Best RMSE: {results_df.iloc[0]['RMSE']:.4f}")

## 5. Model Performance Visualization

In [ ]:
# Performance comparison plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# R² Score comparison
results_df.plot(x='Model', y='R² Score', kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Model R² Score Comparison')
axes[0,0].set_ylabel('R² Score')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# RMSE comparison
results_df.plot(x='Model', y='RMSE', kind='bar', ax=axes[0,1], color='lightcoral')
axes[0,1].set_title('Model RMSE Comparison (Lower is Better)')
axes[0,1].set_ylabel('RMSE')
axes[0,1].tick_params(axis='x', rotation=45)

# MAE comparison
results_df.plot(x='Model', y='MAE', kind='bar', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Model MAE Comparison (Lower is Better)')
axes[1,0].set_ylabel('MAE')
axes[1,0].tick_params(axis='x', rotation=45)

# MAPE comparison
results_df.plot(x='Model', y='MAPE (%)', kind='bar', ax=axes[1,1], color='orange')
axes[1,1].set_title('Model MAPE Comparison (Lower is Better)')
axes[1,1].set_ylabel('MAPE (%)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Prediction vs Actual plots for best models
top_models = results_df.head(4)['Model'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

y_test_original = preprocessor.inverse_transform_target(y_test)

for i, model_name in enumerate(top_models):
    predictions = results[model_name]['predictions']
    r2 = results[model_name]['r2_score']
    rmse = results[model_name]['rmse']
    
    # Scatter plot: Predicted vs Actual
    axes[i].scatter(y_test_original, predictions, alpha=0.6, s=20)
    
    # Perfect prediction line
    min_val = min(y_test_original.min(), predictions.min())
    max_val = max(y_test_original.max(), predictions.max())
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
    
    axes[i].set_xlabel('Actual Values')
    axes[i].set_ylabel('Predicted Values')
    axes[i].set_title(f'{model_name}\nR² = {r2:.3f}, RMSE = {rmse:.3f}')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual plots for best model
best_predictions = results[best_model_name]['predictions']
residuals = y_test_original - best_predictions

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Residuals vs Predicted
axes[0,0].scatter(best_predictions, residuals, alpha=0.6)
axes[0,0].axhline(y=0, color='red', linestyle='--')
axes[0,0].set_xlabel('Predicted Values')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Predicted Values')
axes[0,0].grid(True, alpha=0.3)

# Residuals histogram
axes[0,1].hist(residuals, bins=30, alpha=0.7, color='skyblue')
axes[0,1].set_xlabel('Residuals')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Residuals Distribution')
axes[0,1].grid(True, alpha=0.3)

# Q-Q plot of residuals
stats.probplot(residuals, dist="norm", plot=axes[1,0])
axes[1,0].set_title('Q-Q Plot of Residuals')

# Residuals vs Actual
axes[1,1].scatter(y_test_original, residuals, alpha=0.6)
axes[1,1].axhline(y=0, color='red', linestyle='--')
axes[1,1].set_xlabel('Actual Values')
axes[1,1].set_ylabel('Residuals')
axes[1,1].set_title('Residuals vs Actual Values')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle(f'Residual Analysis - {best_model_name}', fontsize=16)
plt.tight_layout()
plt.show()

# Statistical tests for residuals
from scipy.stats import shapiro, jarque_bera

print(f"Residual Analysis for {best_model_name}:")
print(f"Mean of residuals: {residuals.mean():.6f}")
print(f"Std of residuals: {residuals.std():.4f}")

# Normality test
shapiro_stat, shapiro_p = shapiro(residuals)
print(f"\nShapiro-Wilk normality test:")
print(f"  Statistic: {shapiro_stat:.4f}, p-value: {shapiro_p:.4f}")
print(f"  Residuals are {'normally' if shapiro_p > 0.05 else 'not normally'} distributed")

## 6. Hyperparameter Tuning for Best Model
Optimize the best performing model using grid search.

In [ ]:
# Define hyperparameter grids for different models
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    'Ridge Regression': {
        'alpha': [0.1, 1, 10, 100, 1000]
    },
    'Lasso Regression': {
        'alpha': [0.001, 0.01, 0.1, 1, 10]
    },
    'SVR': {
        'C': [0.1, 1, 10, 100],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2]
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2]
    }
}

# Perform hyperparameter tuning for the best model
if best_model_name in param_grids:
    print(f"Performing hyperparameter tuning for {best_model_name}...")
    
    # Get the model class
    base_model = type(best_model)()
    if hasattr(base_model, 'random_state'):
        base_model.set_params(random_state=42)
    
    # Grid search
    grid_search = GridSearchCV(
        base_model,
        param_grids[best_model_name],
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Best parameters and score
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation RMSE: {np.sqrt(-grid_search.best_score_):.4f}")
    
    # Evaluate tuned model
    tuned_model = grid_search.best_estimator_
    tuned_predictions_transformed = tuned_model.predict(X_test)
    tuned_predictions = preprocessor.inverse_transform_target(tuned_predictions_transformed)
    
    # Calculate metrics for tuned model
    tuned_r2 = r2_score(y_test_original, tuned_predictions)
    tuned_rmse = np.sqrt(mean_squared_error(y_test_original, tuned_predictions))
    
    print(f"\nOriginal {best_model_name} R²: {results[best_model_name]['r2_score']:.4f}")
    print(f"Tuned {best_model_name} R²: {tuned_r2:.4f}")
    print(f"R² Improvement: {tuned_r2 - results[best_model_name]['r2_score']:.4f}")
    
    print(f"\nOriginal {best_model_name} RMSE: {results[best_model_name]['rmse']:.4f}")
    print(f"Tuned {best_model_name} RMSE: {tuned_rmse:.4f}")
    print(f"RMSE Improvement: {results[best_model_name]['rmse'] - tuned_rmse:.4f}")
    
    # Update best model if improved
    if tuned_r2 > results[best_model_name]['r2_score']:
        best_model = tuned_model
        print("\nUpdated best model with tuned parameters!")
    
else:
    print(f"Hyperparameter tuning not implemented for {best_model_name}")
    print("You can add custom parameter grids for this model.")

## 7. Feature Importance Analysis

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    # Get feature names
    feature_names = []
    
    # Numerical features
    numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
    feature_names.extend(numerical_features)
    
    # Categorical features (if any)
    categorical_features = X.select_dtypes(include=['object']).columns.tolist()
    if categorical_features:
        # Get feature names from preprocessor
        if hasattr(preprocessor.preprocessor.named_transformers_['cat'], 'named_steps'):
            encoder = preprocessor.preprocessor.named_transformers_['cat'].named_steps['onehot']
            if hasattr(encoder, 'get_feature_names_out'):
                cat_feature_names = encoder.get_feature_names_out(categorical_features)
                feature_names.extend(cat_feature_names)
    
    # If we couldn't get proper feature names, use generic ones
    if len(feature_names) != X_processed.shape[1]:
        feature_names = [f'Feature_{i}' for i in range(X_processed.shape[1])]
    
    # Create feature importance DataFrame
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Plot feature importance
    plt.figure(figsize=(10, 8))
    top_features = feature_importance_df.head(15)  # Top 15 features
    sns.barplot(data=top_features, y='feature', x='importance')
    plt.title(f'Top Feature Importances - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance_df.head(10))
    
elif hasattr(best_model, 'coef_'):
    # For linear models, show coefficients
    print(f"Model coefficients for {best_model_name}:")
    coefficients = best_model.coef_
    
    # Create feature names (simplified)
    feature_names = [f'Feature_{i}' for i in range(len(coefficients))]
    
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefficients
    }).sort_values('coefficient', key=abs, ascending=False)
    
    # Plot coefficients
    plt.figure(figsize=(10, 8))
    top_coef = coef_df.head(15)
    colors = ['red' if x < 0 else 'blue' for x in top_coef['coefficient']]
    sns.barplot(data=top_coef, y='feature', x='coefficient', palette=colors)
    plt.title(f'Top Coefficients - {best_model_name}')
    plt.xlabel('Coefficient Value')
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Coefficients by Absolute Value:")
    print(coef_df.head(10))
    
else:
    print(f"Feature importance not available for {best_model_name}")

## 8. Learning Curves Analysis
Analyze how model performance changes with training set size.

In [ ]:
from sklearn.model_selection import learning_curve

# Generate learning curves for the best model
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='neg_mean_squared_error'
)

# Convert to RMSE
train_rmse_mean = np.sqrt(-train_scores.mean(axis=1))
train_rmse_std = np.sqrt(train_scores.std(axis=1))
val_rmse_mean = np.sqrt(-val_scores.mean(axis=1))
val_rmse_std = np.sqrt(val_scores.std(axis=1))

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_rmse_mean, 'o-', color='blue', label='Training RMSE')
plt.fill_between(train_sizes, train_rmse_mean - train_rmse_std, 
                 train_rmse_mean + train_rmse_std, alpha=0.2, color='blue')

plt.plot(train_sizes, val_rmse_mean, 'o-', color='red', label='Validation RMSE')
plt.fill_between(train_sizes, val_rmse_mean - val_rmse_std, 
                 val_rmse_mean + val_rmse_std, alpha=0.2, color='red')

plt.xlabel('Training Set Size')
plt.ylabel('RMSE')
plt.title(f'Learning Curves - {best_model_name}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Analysis
final_train_rmse = train_rmse_mean[-1]
final_val_rmse = val_rmse_mean[-1]
gap = final_val_rmse - final_train_rmse

print(f"Learning Curve Analysis for {best_model_name}:")
print(f"Final Training RMSE: {final_train_rmse:.4f}")
print(f"Final Validation RMSE: {final_val_rmse:.4f}")
print(f"Bias-Variance Gap: {gap:.4f}")

if gap < 0.1 * final_val_rmse:
    print("Model appears to have good bias-variance balance.")
elif gap > 0.2 * final_val_rmse:
    print("Model may be overfitting. Consider regularization or more data.")
else:
    print("Model has moderate overfitting. Could benefit from more data or regularization.")

## 9. Model Saving and Loading
Save the best model for future use.

In [ ]:
# Save the best model and preprocessor
model_filename = f'best_regression_model_{best_model_name.lower().replace(" ", "_")}.pkl'
preprocessor_filename = 'regression_preprocessor.pkl'

# Save using joblib (recommended for scikit-learn models)
joblib.dump(best_model, model_filename)
joblib.dump(preprocessor, preprocessor_filename)

print(f"Model saved as: {model_filename}")
print(f"Preprocessor saved as: {preprocessor_filename}")

# Demonstrate loading
loaded_model = joblib.load(model_filename)
loaded_preprocessor = joblib.load(preprocessor_filename)

# Test loaded model
test_predictions_transformed = loaded_model.predict(X_test)
test_predictions = loaded_preprocessor.inverse_transform_target(test_predictions_transformed)
test_r2 = r2_score(y_test_original, test_predictions)

print(f"\nLoaded model R² score: {test_r2:.4f}")
print("Model loading successful!")

## 10. Create Prediction Function
Create a simple function to make predictions on new data.

In [ ]:
def predict_new_data(new_data, model_path=None, preprocessor_path=None):
    """
    Make predictions on new data using the trained model.
    
    Parameters:
    new_data: pandas DataFrame with the same features as training data
    model_path: path to saved model (optional)
    preprocessor_path: path to saved preprocessor (optional)
    
    Returns:
    predictions: array of predicted values in original scale
    """
    
    # Load model and preprocessor if paths provided
    if model_path and preprocessor_path:
        model = joblib.load(model_path)
        prep = joblib.load(preprocessor_path)
    else:
        # Use the current best model and preprocessor
        model = best_model
        prep = preprocessor
    
    # Preprocess the new data
    new_data_processed = prep.transform(new_data)
    
    # Make predictions
    predictions_transformed = model.predict(new_data_processed)
    
    # Transform back to original scale
    predictions = prep.inverse_transform_target(predictions_transformed)
    
    return predictions

def predict_with_confidence(new_data, model_path=None, preprocessor_path=None, n_bootstrap=100):
    """
    Make predictions with confidence intervals using bootstrap sampling.
    
    Parameters:
    new_data: pandas DataFrame with the same features as training data
    model_path: path to saved model (optional)
    preprocessor_path: path to saved preprocessor (optional)
    n_bootstrap: number of bootstrap samples
    
    Returns:
    predictions: mean predictions
    lower_bound: lower confidence bound (2.5%)
    upper_bound: upper confidence bound (97.5%)
    """
    
    # Load model and preprocessor if paths provided
    if model_path and preprocessor_path:
        model_class = joblib.load(model_path).__class__
        prep = joblib.load(preprocessor_path)
    else:
        model_class = best_model.__class__
        prep = preprocessor
    
    # Preprocess the new data
    new_data_processed = prep.transform(new_data)
    
    # Bootstrap predictions
    bootstrap_predictions = []
    
    for i in range(n_bootstrap):
        # Sample with replacement from training data
        indices = np.random.choice(X_train.shape[0], X_train.shape[0], replace=True)
        X_bootstrap = X_train[indices]
        y_bootstrap = y_train[indices]
        
        # Train model on bootstrap sample
        bootstrap_model = model_class()
        if hasattr(bootstrap_model, 'random_state'):
            bootstrap_model.set_params(random_state=42+i)
        
        bootstrap_model.fit(X_bootstrap, y_bootstrap)
        
        # Make predictions
        pred_transformed = bootstrap_model.predict(new_data_processed)
        pred = prep.inverse_transform_target(pred_transformed)
        bootstrap_predictions.append(pred)
    
    # Calculate statistics
    bootstrap_predictions = np.array(bootstrap_predictions)
    mean_predictions = np.mean(bootstrap_predictions, axis=0)
    lower_bound = np.percentile(bootstrap_predictions, 2.5, axis=0)
    upper_bound = np.percentile(bootstrap_predictions, 97.5, axis=0)
    
    return mean_predictions, lower_bound, upper_bound

# Example usage with a sample from the test set
sample_data = X.iloc[:3]  # Take first 3 samples
sample_predictions = predict_new_data(sample_data)

print("Example predictions on sample data:")
print(f"Predictions: {sample_predictions}")
print(f"Actual values: {y.iloc[:3].values}")

print("\nPrediction function created successfully!")

## 11. Summary and Next Steps

### What we accomplished:
1. ✅ Automated data preprocessing pipeline with target transformation
2. ✅ Trained and compared 12 different regression models
3. ✅ Performed hyperparameter tuning on the best model
4. ✅ Comprehensive model evaluation with multiple metrics
5. ✅ Residual analysis and diagnostic plots
6. ✅ Feature importance analysis
7. ✅ Learning curves analysis
8. ✅ Model saving/loading functionality
9. ✅ Prediction functions with confidence intervals

### Key Results:

In [ ]:
# Final summary
print("=" * 60)
print("AUTOMATED REGRESSION ML - FINAL SUMMARY")
print("=" * 60)
print(f"Dataset: {dataset_choice if 'dataset_choice' in locals() else 'Custom dataset'}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")
print(f"Target: {target_name if 'target_name' in locals() else 'Custom target'}")
if preprocessor.target_transformer:
    print(f"Target transformation: {preprocessor.target_transformer}")
print()
print(f"Best Model: {best_model_name}")
print(f"Best R² Score: {results[best_model_name]['r2_score']:.4f}")
print(f"Best RMSE: {results[best_model_name]['rmse']:.4f}")
print(f"Best MAE: {results[best_model_name]['mae']:.4f}")
print(f"Best MAPE: {results[best_model_name]['mape']:.2f}%")
print()
print("Top 3 Models by R² Score:")
for i, row in results_df.head(3).iterrows():
    print(f"  {i+1}. {row['Model']}: R² = {row['R² Score']:.4f}, RMSE = {row['RMSE']:.4f}")
print()
print("Files saved:")
print(f"  - {model_filename}")
print(f"  - {preprocessor_filename}")
print("=" * 60)

# Model interpretation
r2_score_best = results[best_model_name]['r2_score']
print("\n📊 MODEL INTERPRETATION:")
if r2_score_best > 0.8:
    print("🟢 Excellent model performance! The model explains most of the variance.")
elif r2_score_best > 0.6:
    print("🟡 Good model performance. The model captures most patterns in the data.")
elif r2_score_best > 0.4:
    print("🟠 Moderate model performance. Consider feature engineering or different algorithms.")
else:
    print("🔴 Poor model performance. The data may need more preprocessing or different approach.")

# Provide Google Colab specific tips
print("\n📝 GOOGLE COLAB TIPS:")
print("1. To use your own dataset, upload a CSV file using the file upload cell")
print("2. Save important files to Google Drive to persist them")
print("3. Use GPU runtime for faster training on large datasets")
print("4. Consider feature engineering for better performance")
print("5. Try ensemble methods or neural networks for complex problems")
print("6. Use the confidence prediction function for uncertainty quantification")